In [18]:
import pandas as pd

In [19]:
# Data Loading

In [20]:
#Data Loading
per = pd.read_csv("DAC18PER.csv")
hh = pd.read_csv("DAC18HH.csv")

print("Person file:", per.shape)
print("Household file:", hh.shape)

Person file: (65487, 2049)
Household file: (21949, 100)


In [21]:
#Person-level variables
person_vars = [
    # Identifiers / construction-
    "ABSPID",
    "ABSFID",
    "ABSHID",
    "ABSIID",
    "POPESTAB",
    "RECLPSN",

    # Survey design
    "FINWTP",

    # Predisposing

    "AGEPC",
    "SEX",
    "EDLVLATB",

    # Enabling

    "ARIAC",
    "SPENGPRO",


    # Psychosocial

    "SOCMORCO",
    "K10CATEG",


    # Need

    "DISBSTAT",
    "SFHEALTH",

    # Domain-specific informal assistance received

    "RASFRICO",
    "RASFRIMO",
    "RASFRISC",
    "RASFRIEM",
    "RASFRIHC",
    "RASFRIHO",
    "RASFRIME",
    "RASFRIPM",
    "RASFRIPA",
    "RASFRITR",
    
    # Domain-specific assistance need and need met
    "WHNASMOB",
    "RASEXMOB",
    "WHNASSCR",
    "RASEXSC",
    "WHNASCOM",
    "RASEXCOM",
    "NASSKINC",
    "RASEXHC",
    "WHNASEMO",
    "RASEXGUI",
    "WHNASHOM",
    "RASEXHOM",
    "WHNASPRO",
    "RASEXPRP",
    "WNASMEAL",
    "RASEXMEA",
    "WHNASFIN",
    "RASEXPAP",
    "NASTRANS",
    "RASEXTRA",
    "NAHCFCAR"
]

# Social Participation (multiple response)

soc_away_cols = [
    "SOC3AWYA",
    "SOC3AWYB",
    "SOC3AWYC",
    "SOC3AWYD",
    "SOC3AWYE",
    "SOC3AWYF",
    "SOC3AWYG",
    "SOC3AWYH",
    "SOC3AWYI",
    "SOC3AWYJ",
    "SOC3AWYK",
    "SOC3AWYL",
    "SOC3AWYM",
    "SOC3AWYN",
]

soc_home_cols = [
    "SOCL3MA",
    "SOCL3MB",
    "SOCL3MC",
    "SOCL3MD",
    "SOCL3ME",
    "SOCL3MF",
    "SOCL3MG",
    "SOCL3MH",
]

person_vars += soc_away_cols
person_vars += soc_home_cols

In [22]:
#Replicate Person weights

replicate_weights = [
    f"WPM{i:04d}"
    for i in range(101, 161)
]

person_vars += replicate_weights

In [23]:
per_filtered = per[
    (per["POPESTAB"] == 2) &
    (per["AGEPC"].isin([25, 26, 27, 28, 29]))
].copy()

print("Original person file:", per.shape)
print("65+ household population:", per_filtered.shape)

Original person file: (65487, 2049)
65+ household population: (9271, 2049)


In [24]:
existing_person_vars = [
    col for col in person_vars
    if col in per.columns
]

person_selected = per_filtered[existing_person_vars].copy()

print("EXtracted person data:", person_selected.shape)

EXtracted person data: (9271, 129)


In [25]:
household_vars = [
    "ABSHID",      
    "HHNOPSNB",    
    "INCDECHD",    
]

missing_household = [
    col for col in household_vars
    if col not in hh.columns
]
household_selected = hh[household_vars].copy()

In [28]:
print(len(household_selected))
print(household_selected["ABSHID"].nunique()
)
print(household_selected["ABSHID"].duplicated().sum())

21949
21949
0


In [29]:
df = person_selected.merge(
    household_selected,
    on="ABSHID",
    how="left",
    validate="many_to_one",
    indicator=True
)
print(df["_merge"].value_counts(dropna=False))

_merge
both          9271
left_only        0
right_only       0
Name: count, dtype: int64


In [30]:
print(df.shape)


(9271, 132)


In [31]:
need_cols_to_remove = [
    "ANY_ASSISTANCE_NEED"
    ""
]

df = df.drop(
    columns=need_cols_to_remove,
    errors="ignore"
)

In [32]:
df.to_csv(
    "sdac2018_candidate_variables_raw.csv",
    index=False
)